In [1]:
import torch
from src.Diffusion2d import Unet2D

D, S, C = 7, 288, 1
kw = dict(dim=32, dim_mults=(1,2,4), resnet_block_groups=8, channels=C,
          cat_cardinalities=[3,5], n_cont=2, n_bool=1, n_dyn_cond_cols=4,
          n_days=D, steps_per_day=S)
m = Unet2D(**kw)
x   = torch.randn(2, C, D*S)
t   = torch.randint(0, 1000, (2,))
cat = torch.zeros(2, 2, dtype=torch.long)
out = m(x, t, cat, torch.randn(2,2), torch.randn(2,1), torch.randn(2,4,D*S))
assert out.shape == x.shape
print(sum(p.numel() for p in m.parameters())/1e6, "M params")

1.387681 M params


In [3]:
from src.Diffusion1d import DiffusionSchedule, p_losses
dyn = torch.randn(2, 4, D*S)   # (batch, n_dyn_cond_cols, window_length)
sch = DiffusionSchedule.make("sigmoid", 1000)
t = torch.randint(0, 1000, (2,))
loss = p_losses(m, sch, x, t, cat, torch.randn(2,2), torch.randn(2,1), dyn=dyn)
loss.backward()

In [5]:
m.init_conv.weight.grad.abs().sum() > 0

tensor(True)

In [6]:
from src.Diffusion1d import Unet1D
kw1d = {k: v for k, v in kw.items() if k not in ("n_days", "steps_per_day")}
n = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
print("1D", n(Unet1D(**kw1d)), "2D", n(Unet2D(**kw)))

1D 0.638817 2D 1.387681
